# Ejercicio 1 — Cobertura de Djokovic: preguntas 3 y 4

**Jugador objetivo:** Novak Djokovic  
**Fuente:** JeffSackmann/tennis_MatchChartingProject (CC BY-NC-SA 4.0)  
**Objetivo:** Calcular KPIs de cobertura para responder las preguntas 3 y 4 del P1.


In [1]:
import pandas as pd
import os

# ─── Ajusta esta ruta si es necesario ───────────────────────────────────────
DATA_DIR = "../data"
# ─────────────────────────────────────────────────────────────────────────────

MATCHES_FILE = os.path.join(DATA_DIR, "charting-m-matches.csv")
print(f"Ruta: {os.path.abspath(MATCHES_FILE)}")
print(f"Existe: {os.path.exists(MATCHES_FILE)}")

Ruta: c:\Users\pater\Desktop\DJOKOVIC\data\charting-m-matches.csv
Existe: True


In [2]:
# Carga con fallback de encoding
try:
    df = pd.read_csv(MATCHES_FILE, encoding="utf-8", low_memory=False)
except UnicodeDecodeError:
    df = pd.read_csv(MATCHES_FILE, encoding="latin-1", low_memory=False)

print(f"Filas totales cargadas : {len(df):,}")
print(f"Columnas               : {list(df.columns)}")

Filas totales cargadas : 7,566
Columnas               : ['match_id', 'Player 1', 'Player 2', 'Pl 1 hand', 'Pl 2 hand', 'Date', 'Tournament', 'Round', 'Time', 'Court', 'Surface', 'Umpire', 'Best of', 'Final TB?', 'Charted by']


In [3]:
# Eliminar fila corrupta documentada en el diccionario (fila 907)
ID_CORRUPTO = "20240915-M-Davis_Cup_World_Group-RR-Botic_Van_De_Zandschulp-Matteo_Berrettini"
df = df[df["match_id"] != ID_CORRUPTO].copy()
print(f"Filas tras eliminar fila corrupta: {len(df):,}")

Filas tras eliminar fila corrupta: 7,564


In [4]:
# Filtrar partidos de Djokovic (puede ser Player 1 o Player 2)
djok = df[
    (df["Player 1"] == "Novak Djokovic") |
    (df["Player 2"] == "Novak Djokovic")
].copy()

# Columna Rival
djok["Rival"] = djok.apply(
    lambda r: r["Player 2"] if r["Player 1"] == "Novak Djokovic" else r["Player 1"],
    axis=1
)

# Columna Año (Date viene como AAAAMMDD string)
djok["Año"] = pd.to_numeric(
    djok["Date"].astype(str).str.strip().str[:4],
    errors="coerce"
).astype("Int64")

print(f"Partidos de Djokovic encontrados: {len(djok):,}")
djok[["match_id", "Player 1", "Player 2", "Rival", "Año", "Surface", "Tournament"]].head(5)

Partidos de Djokovic encontrados: 553


,match_id,Player 1,Player 2,Rival,Año,Surface,Tournament
59,20260311-M-Indian_Wells_Masters-R16-Novak_Djok...,Novak Djokovic,Jack Draper,Jack Draper,2026,Hard,Indian Wells Masters
65,20260309-M-Indian_Wells_Masters-R32-Aleksandar...,Aleksandar Kovacevic,Novak Djokovic,Aleksandar Kovacevic,2026,Hard,Indian Wells Masters
123,20260201-M-Australian_Open-F-Novak_Djokovic-Ca...,Novak Djokovic,Carlos Alcaraz,Carlos Alcaraz,2026,Hard,Australian Open
125,20260130-M-Australian_Open-SF-Jannik_Sinner-No...,Jannik Sinner,Novak Djokovic,Jannik Sinner,2026,Hard,Australian Open
127,20260128-M-Australian_Open-QF-Novak_Djokovic-L...,Novak Djokovic,Lorenzo Musetti,Lorenzo Musetti,2026,Hard,Australian Open


---
## Pregunta 3 — Distribución de partidos de Djokovic

In [5]:
# KPIs globales
n_partidos = len(djok)
n_rivales  = djok["Rival"].nunique()
n_torneos  = djok["Tournament"].nunique()
superficies = sorted(djok["Surface"].dropna().unique().tolist())

print(f"Partidos de Djokovic charteados : {n_partidos}")
print(f"Rivales únicos                  : {n_rivales}")
print(f"Torneos únicos                  : {n_torneos}")
print(f"Superficies disponibles         : {superficies}")

Partidos de Djokovic charteados : 553
Rivales únicos                  : 161
Torneos únicos                  : 51
Superficies disponibles         : ['Clay', 'Grass', 'Hard']


In [6]:
# Distribución por superficie
sup = (
    djok["Surface"]
    .value_counts()
    .rename_axis("Superficie")
    .reset_index(name="Partidos")
)
sup["% del total"] = (sup["Partidos"] / n_partidos * 100).round(1)
print("--- Distribución por SUPERFICIE ---")
print(sup.to_string(index=False))

--- Distribución por SUPERFICIE ---
Superficie  Partidos  % del total
      Hard       366         66.2
      Clay       129         23.3
     Grass        58         10.5


In [7]:
# Distribución por año (todos los años)
por_año = (
    djok.groupby("Año", dropna=True)
    .size()
    .rename("Partidos")
    .reset_index()
    .sort_values("Año")
)
print("--- Distribución por AÑO ---")
print(por_año.to_string(index=False))

--- Distribución por AÑO ---
 Año  Partidos
2005         2
2006         5
2007        22
2008        21
2009        22
2010        18
2011        25
2012        41
2013        33
2014        20
2015        51
2016        28
2017        14
2018        25
2019        28
2020        12
2021        29
2022        31
2023        45
2024        38
2025        37
2026         6


In [8]:
# Top 20 torneos
top_torneos = (
    djok["Tournament"]
    .value_counts()
    .head(20)
    .rename_axis("Torneo")
    .reset_index(name="Partidos")
)
print("--- Top 20 TORNEOS ---")
print(top_torneos.to_string(index=False))

--- Top 20 TORNEOS ---
              Torneo  Partidos
     Australian Open        62
             US Open        51
           Wimbledon        48
         Tour Finals        44
       Roland Garros        40
        Rome Masters        32
 Monte Carlo Masters        28
       Paris Masters        24
    Shanghai Masters        23
       Miami Masters        22
Indian Wells Masters        19
            Olympics        17
               Dubai        17
  Cincinnati Masters        16
      Canada Masters        14
      Madrid Masters        14
             Beijing        10
                Doha         9
              Astana         5
               Paris         5


In [9]:
# Top 20 rivales
top_rivales = (
    djok["Rival"]
    .value_counts()
    .head(20)
    .rename_axis("Rival")
    .reset_index(name="Partidos")
)
print("--- Top 20 RIVALES ---")
print(top_rivales.to_string(index=False))

--- Top 20 RIVALES ---
                Rival  Partidos
         Rafael Nadal        52
        Roger Federer        47
          Andy Murray        30
        Tomas Berdych        15
Juan Martin Del Potro        12
   Stefanos Tsitsipas        11
         Gael Monfils        11
        Stan Wawrinka        11
   Jo Wilfried Tsonga        10
        Jannik Sinner        10
      Daniil Medvedev        10
        Kei Nishikori        10
         David Ferrer        10
      Lorenzo Musetti         9
        Dominic Thiem         9
      Grigor Dimitrov         8
       Hubert Hurkacz         8
          Marin Cilic         8
      Richard Gasquet         7
       Carlos Alcaraz         7


---
## Pregunta 4 — Cobertura temporal

In [10]:
primer_año   = int(djok["Año"].min())
ultimo_año   = int(djok["Año"].max())
n_temporadas = int(djok["Año"].nunique())

print(f"Primer año con datos : {primer_año}")
print(f"Último año con datos : {ultimo_año}")
print(f"Temporadas cubiertas : {n_temporadas}  (años distintos con >= 1 partido)")
print(f"Rango total (años)   : {ultimo_año - primer_año + 1}")

# Huecos en la serie temporal
años_con_datos = set(djok["Año"].dropna().astype(int))
años_rango     = set(range(primer_año, ultimo_año + 1))
huecos         = sorted(años_rango - años_con_datos)
if huecos:
    print(f"Años sin partidos    : {huecos}")
else:
    print("Sin huecos — cobertura continua año a año")

Primer año con datos : 2005
Último año con datos : 2026
Temporadas cubiertas : 22  (años distintos con >= 1 partido)
Rango total (años)   : 22
Sin huecos — cobertura continua año a año


In [11]:
# Últimos 5 años — para razonar la ventana reciente
print(f"--- Partidos últimos 5 años ({ultimo_año - 4}–{ultimo_año}) ---")
recientes = por_año[por_año["Año"] >= ultimo_año - 4]
print(recientes.to_string(index=False))

--- Partidos últimos 5 años (2022–2026) ---
 Año  Partidos
2022        31
2023        45
2024        38
2025        37
2026         6
